In [ ]:
# This exercise is designed to use pytorch, understand with a LOrA is and to use it in a LLM model, in a more advanced way.
# At the end of the exercise, we use LLMs to produce sentences using an open-source causal LLM from Facebook
# The exercise has been done with the help of Gemini, 22. June 2026

In [5]:
import torch

In [6]:
# Creating tensors
X = torch.tensor([[1.0], [2.0], [3.0], [4.0]], dtype=torch.float32)
Y = torch.tensor([[2.0], [4.0], [6.0], [8.0]], dtype=torch.float32)

In [7]:
# Define a normal numpy array an turn it into a tensor
import numpy as np
np_array = np.array([[1.0], [2.0], [3.0], [4.0]])
Z = torch.from_numpy(np_array)


In [8]:
# Design a model using torch.nn commenting each section of the code
import torch.nn as nn

class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        # Define a linear layer with 1 input feature and 1 output feature
        # A simple linear layer: y = w*x + b
        self.linear = nn.Linear(in_features=1, out_features=1)

    def forward(self, x):
        # Define the forward pass of the model
        return self.linear(x)

model = LinearRegressionModel()

In [9]:
# Define loss and optimizer 
loss_fn = nn.MSELoss()  # Mean Squared Error loss
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # Stochastic, with lr being learning rate

In [10]:
# Training loop
epochs = 1000

for epoch in range(epochs):
    # Forward pass: compute predicted y by passing x to the model
    Y_pred = model(X)

    # Compute and print loss
    loss = loss_fn(Y_pred, Y)
    
    # Zero gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [100/1000], Loss: 0.1387
Epoch [200/1000], Loss: 0.0761
Epoch [300/1000], Loss: 0.0418
Epoch [400/1000], Loss: 0.0229
Epoch [500/1000], Loss: 0.0126
Epoch [600/1000], Loss: 0.0069
Epoch [700/1000], Loss: 0.0038
Epoch [800/1000], Loss: 0.0021
Epoch [900/1000], Loss: 0.0011
Epoch [1000/1000], Loss: 0.0006


In [11]:
model.eval() # Set model to evaluation mode
with torch.inference_mode(): # Context manager that disables gradient tracking
    new_prediction = model(torch.tensor([[5.0]]))
    print(f"Prediction for x=5: {new_prediction.item()}") # Should be close to 10.0

Prediction for x=5: 9.957162857055664


# Now with two parameters 
# Let's predict the price of a flat knowing the trend 

In [12]:
import torch
import torch.nn as nn

# Features: [Square Footage (normalized), Bedrooms]
X = torch.tensor([
    [1.5, 2.0],
    [2.0, 3.0],
    [2.5, 3.0],
    [3.0, 4.0],
    [3.5, 4.0],
    [4.0, 5.0]
], dtype=torch.float32)

# Targets: Price in hundreds of thousands of dollars
y = torch.tensor([
    [2.5],
    [3.2],
    [3.7],
    [4.5],
    [5.0],
    [5.8]
], dtype=torch.float32)

In [13]:
# Custom subclass of nn.Module for the model
class HousePriceModel(nn.Module):
    def __init__(self):
        super(HousePriceModel, self).__init__()
        # Define linear layer with 2 inputs and 1 output
        self.linear = nn.Linear(in_features = 2, out_features = 1)
    def forward(self, x):
        # Forward pass through the linear layer
        return self.linear(x)

model = HousePriceModel()

In [14]:
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.05)

In [15]:
epochs = 500
# Training loop
for epoch in range(epochs):
    # Forward pass: compute predicted y by passing x to the model
    y_pred = model(X)

    # Compute and print loss
    loss = loss_function(y_pred, y)
    
    # Zero gradients, perform a backward pass, and update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [16]:
model.eval() # Set model to evaluation mode 

with torch.inference_mode(): # Context manager that disables gradient tracking
    new_prediction = model(torch.tensor([[3.2 , 4.0]]))
    print(f"Price predicted for [3.2 sq ft, 4.0 bedrooms]: {new_prediction.item()} hundred thousand dollars") # Should be close to 10.0

Price predicted for [3.2 sq ft, 4.0 bedrooms]: 4.701900482177734 hundred thousand dollars


# Now, we move to more advanced problems

In [17]:
# Advanced Production Design (torch and amp)
import torch

class AdvancedTrainer:
    def __init__(self, model, optimizer, loss_fn):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        
        # 1. Compile the model graph. Crucial for 2026 optimization.
        # This optimizes the underlying execution graph and removes Python overhead.
        self.compiled_model = torch.compile(model)
        
        # 2. Initialize a GradScaler for AMP (prevents small gradients from underflowing to 0)
        self.scaler = torch.amp.GradScaler('cuda')

    def train_step(self, X, y):
        self.optimizer.zero_grad(set_to_none=True) # set_to_none=True optimizes memory allocation
        
        # Automatic Mixed Precision context block
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            predictions = self.compiled_model(X)
            loss = self.loss_fn(predictions, y)
            
        # Scales loss and calls backward() to create gradients safely
        self.scaler.scale(loss).backward()
        
        # Unscales gradients and steps the optimizer
        self.scaler.step(self.optimizer)
        self.scaler.update()
        
        return loss.item()

In [18]:
# Structural Tensor Control via torch.einsum

# Multi-head attention matrix multiplications across Batch ($B$), Heads ($H$), Sequence Length ($S$), and Dimension ($D$) properties:
# Batch=16, Heads=8, Seq_Len=64, Dim=64
Q = torch.randn(16, 8, 64, 64)
K = torch.randn(16, 8, 64, 64)

# Traditional messy way: Q.transpose(1, 2) @ K.transpose(1, 2).transpose(-2, -1)...
# Clean, self-documenting 2026 approach:
attention_scores = torch.einsum('bhqd, bhkd -> bhqk', Q, K)

# The string format explicitly shows:
# (Batch, Heads, Query_len, Dim) x (Batch, Heads, Key_len, Dim) -> (Batch, Heads, Query_len, Key_len)

In [19]:
# Advanced exercise: write a Custom LoRA Layer
# Low-Rank Adaptation (LoRA) is the industry-standard method for parameter-efficient fine-tuning of Large Language Models and Vision Transformers

import math

class LoraLinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 4, lora_alpha: int = 32):
        super().__init__()

        # Original pre-trained massive linear layer
        self.pretrained_linear = base_layer

        # Now, we freeze the pretrained weights. We dont want to compute gradients for them during fine-tuning.
        # This is the key. We do not track history of gradients so the execution is much faster and memory efficient.
        for param in self.pretrained_linear.parameters():
            param.requires_grad = False

        in_features = base_layer.in_features
        out_features = base_layer.out_features

        # Low-Rank Adaptation Path
        # Matrix A: Projects down from in_features -> rank 'r'
        self.lora_A = nn.Linear(in_features, r, bias=False)

        # Matrix B : Projects up from rank 'r' -> out_features
        self.lora_B = nn.Linear(r, out_features, bias=False)

        # Scaling factors for LoRA. This is a hyperparameter that can be tuned.
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r

        self.reset_lora_parameters()

    def reset_lora_parameters(self):
        # Initialize LoRA weights to small values to ensure they don't dominate the pretrained weights initially.
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))

        # Matrix B MUST be initialized to zero! 
        # This ensures that on Step 1 of training, the LoRA path outputs exactly 0,
        # meaning the model behaves identically to the original frozen model.
        nn.init.zeros_(self.lora_B.weight)

    # the forward function defines the actual computational graph—the path data takes when you pass an input tensor through your model.
    def forward(self, x: torch.Tensor) -> torch.Tensor: 
        # Base Path: Calculate the original frozen outputs
        base_output = self.pretrained_linear(x)
        
        # Adapter Path: Pass through A (down), then B (up), and apply scaling
        # This line executes from the inside out. It compresses and decompresses the data through a geometric bottleneck to capture new patterns.
        lora_output = self.lora_B(self.lora_A(x)) * self.scaling
        
        # Fuse them together
        return base_output + lora_output

    



# REAL LIFE CASE 
We will use transformers for it and an open source LLM from facebook 

In [20]:
# Import the Hugging Face Transformers library for working with pre-trained models and tokenizers
from transformers import AutoModelForCausalLM, AutoTokenizer

/opt/miniconda3/envs/llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
import math
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==========================================
# STEP 1: Define your Custom LoRA Layer
# ==========================================
class LoraLinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, r: int = 4, lora_alpha: int = 16):
        super().__init__()
        # Instead of creating a new linear layer, we take the original LLM's layer
        self.pretrained_linear = base_layer
        
        # Freeze the original LLM layer parameters completely
        for param in self.pretrained_linear.parameters():
            param.requires_grad = False
            
        in_features = base_layer.in_features
        out_features = base_layer.out_features

        # Detect the data type of the original layer to ensure our LoRA layers match it
        base_dtype = base_layer.weight.dtype
        
        # Create our low-rank trainable adapter tracking paths
        self.lora_A = nn.Linear(in_features, r, bias=False).to(base_dtype)
        self.lora_B = nn.Linear(r, out_features, bias=False).to(base_dtype)
        
        self.scaling = lora_alpha / r
        
        self.reset_lora_parameters()

    def reset_lora_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Run through the original frozen LLM layer path
        base_output = self.pretrained_linear(x)
        # Run through our active custom LoRA path
        lora_output = self.lora_B(self.lora_A(x)) * self.scaling
        return base_output + lora_output


# ==========================================
# STEP 2: The Advanced Python Injection Logic
# ==========================================
def inject_lora_layers(model, target_layers=["q_proj", "v_proj"], r=4, lora_alpha=16):
    """
    Traverses the LLM structure recursively. If it finds a layer name matching 
    our target attention projections, it replaces it with our LoraLinear layer.
    """
    # Get a list of all sub-modules inside the LLM
    for name, module in model.named_modules():
        # We target query (q_proj) and value (v_proj) matrices inside LLM attention blocks
        if any(name.endswith(target) for target in target_layers):
            
            # Navigate to the parent module holding this layer using string parsing
            parts = name.split(".")
            parent = model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            
            # Fetch the original linear layer
            base_layer = getattr(parent, parts[-1])
            
            if isinstance(base_layer, nn.Linear):
                # Wrap it with our custom LoraLinear layer!
                new_lora_layer = LoraLinear(base_layer, r=r, lora_alpha=lora_alpha)
                # Overwrite the layer inside the LLM dynamically
                setattr(parent, parts[-1], new_lora_layer)
                print(f"Successfully injected LoRA into: {name}")


# ==========================================
# STEP 3: Execution and Verification Pipeline
# ==========================================
if __name__ == "__main__":
    # 1. Load a tiny, real, open-source causal LLM from Facebook
    model_name = "facebook/opt-125m"
    print(f"Loading open-source LLM: {model_name}...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    llm = AutoModelForCausalLM.from_pretrained(model_name)

    print("\n--- Injecting Custom LoRA Modules ---")
    # 2. Inject our custom code right into the attention projections of the LLM
    inject_lora_layers(llm, target_layers=["q_proj", "v_proj"], r=4, lora_alpha=16)

    # 3. Calculate and display our optimization parameters
    total_params = 0
    trainable_params = 0
    
    for param in llm.parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    print("\n=== SYSTEM ARCHITECTURE STATS ===")
    print(f"Total Parameters in LLM:  {total_params:,}")
    print(f"Trainable Parameters:      {trainable_params:,}")
    print(f"Percentage to train:       {(trainable_params / total_params) * 100:.4f}%")

    # Run a real text tokenization generation through your modified LLM!
    input_text = "The future of open source machine learning is"
    inputs = tokenizer(input_text, return_tensors="pt")
    
    print(f"\nPassing text through modified LLM structure...")
    with torch.no_grad():
        outputs = llm.generate(**inputs, max_new_tokens=30)
        
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nModel Output:\n-> \"{generated_text}\"")

Loading open-source LLM: facebook/opt-125m...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 20188.57it/s]



--- Injecting Custom LoRA Modules ---
Successfully injected LoRA into: model.decoder.layers.0.self_attn.v_proj
Successfully injected LoRA into: model.decoder.layers.0.self_attn.q_proj
Successfully injected LoRA into: model.decoder.layers.1.self_attn.v_proj
Successfully injected LoRA into: model.decoder.layers.1.self_attn.q_proj
Successfully injected LoRA into: model.decoder.layers.2.self_attn.v_proj
Successfully injected LoRA into: model.decoder.layers.2.self_attn.q_proj
Successfully injected LoRA into: model.decoder.layers.3.self_attn.v_proj
Successfully injected LoRA into: model.decoder.layers.3.self_attn.q_proj
Successfully injected LoRA into: model.decoder.layers.4.self_attn.v_proj
Successfully injected LoRA into: model.decoder.layers.4.self_attn.q_proj
Successfully injected LoRA into: model.decoder.layers.5.self_attn.v_proj
Successfully injected LoRA into: model.decoder.layers.5.self_attn.q_proj
Successfully injected LoRA into: model.decoder.layers.6.self_attn.v_proj
Successfully

In [ ]:
# Try with more complicated text. 

input_text = "In a village of La Mancha, the name of which I have no desire to call to mind, there lived not long since one of those gentlemen"
inputs = tokenizer(input_text, return_tensors="pt")

print(f"\nPassing text through modified LLM structure...")
with torch.no_grad():
    outputs = llm.generate(**inputs, max_new_tokens=100)
    
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nModel Output:\n-> \"{generated_text}\"")


Passing text through modified LLM structure...

Model Output:
-> "n a village of La Mancha, the name of which I have no desire to call to mind, there lived not long since one of those gentlemen, who had been a priest, had been killed by a mob of the people.

The village was called La Mancha, and the name of the village was La Mancha.

The name of the village was La Mancha.

The name of the village was La Mancha.

The name of the village was La Mancha.

The name of the village was La Mancha.

The name of the village was La Mancha.

"


In [ ]:
# VERY BAD RESULTS. We go back to the same sentence over and over again. The model is not able to generate new sentences. 

In [30]:

input_text = "In the beginning God created the heaven and the"
inputs = tokenizer(input_text, return_tensors="pt")

print(f"\nPassing text through modified LLM structure...")
with torch.no_grad():
    outputs = llm.generate(**inputs, max_new_tokens=100)
    
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\nModel Output:\n-> \"{generated_text}\"")


Passing text through modified LLM structure...

Model Output:
-> "In the beginning God created the heaven and the earth.

In the beginning God created the earth and the heavens.

In the beginning God created the heavens and the earth.

In the beginning God created the heavens and the earth.

In the beginning God created the heavens and the earth.

In the beginning God created the heavens and the earth.

In the beginning God created the heavens and the earth.

In the beginning God created the heavens and the earth.

In the beginning God created"
